In [ ]:
import numpy as np
import os

era5_path = "/home/teoaivalis/floods_kg/reanalysis_data/era5_cnn_256_baseline.npy"
pangu_path = "/home/teoaivalis/floods_kg/reanalysis_data/pangu_cnn_256_baseline.npy"
graphcast_path = "/home/teoaivalis/floods_kg/reanalysis_data/graphcast_cnn_256_baseline.npy"

era5_data = np.load(era5_path, allow_pickle=True).item()
pangu_data = np.load(pangu_path, allow_pickle=True).item()
graphcast_data = np.load(graphcast_path, allow_pickle=True).item()

In [ ]:
datasets = {
    "ERA5 (Truth)": era5_data,
    "Pangu (Pred)": pangu_data,
    "GraphCast (Pred)": graphcast_data
}

for name, data in datasets.items():
    num_events = len(data)
    first_key = list(data.keys())[0]
    vector_shape = data[first_key].shape
    
    print(f"--- {name} ---")
    print(f"Total Flood Events: {num_events}")
    print(f"Embedding Dimension: {vector_shape}")
    print(f"Sample Event ID: {first_key}\n")

In [ ]:
def print_dict_head(data, n=3):
    keys = list(data.keys())[:n]
    for i, key in enumerate(keys):
        vector_sample = data[key][:5] 
        print(f"{i+1}. Event ID: {key}")
        print(f"   Vector Slice (first 5 dims): {vector_sample}...")
    print("-" * 30)

print(" ERA5 Head:")
print_dict_head(era5_data)

print("Pangu Head:")
print_dict_head(pangu_data)

print(" GraphCast Head:")
print_dict_head(graphcast_data)

In [ ]:
import torch
import numpy as np
import torch.nn.functional as F

def check_embedding_stats(name, data_dict):
    embeddings = torch.tensor(np.array(list(data_dict.values())))
    
    var = torch.var(embeddings, dim=0).mean().item()
    
    norms = torch.norm(embeddings, p=2, dim=1).mean().item()
    
    sample_size = min(500, embeddings.shape[0])
    sample = embeddings[:sample_size]
    sample_norm = F.normalize(sample, p=2, dim=1)
    cos_sim = torch.mm(sample_norm, sample_norm.t())
    
    avg_sim = (cos_sim.sum() - sample_size) / (sample_size * (sample_size - 1))

    print(f"--- Statistics for {name} ---")
    print(f"Mean Variance: {var:.6f}")
    print(f"Average L2 Norm: {norms:.2f}")
    print(f"Avg Internal Cosine Similarity: {avg_sim:.4f}")

check_embedding_stats("ERA5", era5_data)
check_embedding_stats("Pangu", pangu_data)
check_embedding_stats("GraphCast", graphcast_data)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np


era5_emb = np.load("/home/teoaivalis/floods_kg/reanalysis_data/era5_cnn_256_baseline.npy", allow_pickle=True).item()
pangu_emb = np.load("/home/teoaivalis/floods_kg/reanalysis_data/pangu_cnn_256_baseline.npy", allow_pickle=True).item()
gc_emb = np.load("/home/teoaivalis/floods_kg/reanalysis_data/graphcast_cnn_256_baseline.npy", allow_pickle=True).item()

def find_top_n_era5_matches(query_id, pangu_dict, era5_dict, n=5):
    if query_id not in pangu_dict:
        return

    query_vec = torch.tensor(pangu_dict[query_id]).float().unsqueeze(0) # [1, 1024]
    
    era5_keys = list(era5_dict.keys())
    era5_matrix = torch.tensor(np.array([era5_dict[k] for k in era5_keys])).float() # [N, 1024]
    
    query_norm = F.normalize(query_vec, p=2, dim=1)
    era5_norm = F.normalize(era5_matrix, p=2, dim=1)
    
    similarities = torch.mm(query_norm, era5_norm.t()).squeeze(0) # [N]
    
    top_values, top_indices = torch.topk(similarities, k=n)
    
    print(f"Query (Pangu Source): {query_id}")
    print(f"--- Top {n} Most Similar Events in ERA5 (Ground Truth Source) ---")
    
    for i in range(n):
        match_id = era5_keys[top_indices[i]]
        score = top_values[i].item()
        
        print(f"{i+1}. {match_id} | Similarity: {score:.6f}")

print ("Pangu Predictions")
find_top_n_era5_matches("FL_2018_000425_LKA", pangu_emb, era5_emb)
find_top_n_era5_matches("FL_2020_000128_KEN", pangu_emb, era5_emb)
find_top_n_era5_matches("FL_2021_000161_IND", pangu_emb, era5_emb)
find_top_n_era5_matches("FL_2022_000201_ZAF", pangu_emb, era5_emb)

print ("GraphCast Predictions")
find_top_n_era5_matches("FL_2019_000162_MYS", pangu_emb, gc_emb)
find_top_n_era5_matches("FL_2020_000128_KEN", pangu_emb, gc_emb)
find_top_n_era5_matches("FL_2020_000161_BGD", pangu_emb, gc_emb)
find_top_n_era5_matches("FL_2020_000172_MMR", pangu_emb, gc_emb)


visualise for the paper

In [ ]:
!pip install cartopy xarray scipy netCDF4 geopandas

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

file_path = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth/FF_2004_000060_VNM/center_ERA5_truth.csv"
df = pd.read_csv(file_path)

df['temp_c'] = df['temperature'] - 273.15
df['temp_2m_c'] = df['2m_temperature'] - 273.15

plt.style.use('seaborn-v0_8-paper')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6), dpi=300)


v_profile = df.groupby('level')[['temp_c', 'specific_humidity', 'wind_speed']].mean().reset_index()
v_profile = v_profile.sort_values('level', ascending=False) # 1000hPa at bottom

ax1.plot(v_profile['temp_c'], v_profile['level'], 'r-o', label='Temp (°C)', linewidth=2)
ax1_twin = ax1.twiny()
ax1_twin.plot(v_profile['specific_humidity'], v_profile['level'], 'g--s', label='Spec. Humidity', alpha=0.7)

ax1.set_ylabel('Pressure Level (hPa)', fontsize=12)
ax1.set_xlabel('Temperature (°C)', color='r', fontsize=12)
ax1_twin.set_xlabel('Specific Humidity (kg/kg)', color='g', fontsize=12)
ax1.invert_yaxis() 
ax1.set_title('A: Vertical Atmospheric Structure', fontsize=14, fontweight='bold')
ax1.grid(True, linestyle=':', alpha=0.6)

sns.scatterplot(data=df[df['level'] == 1000], x='surface_pressure', y='precipitation_mm_24h', 
                hue='10m_wind_speed', size='10m_wind_speed', palette='viridis', ax=ax2)

ax2.set_title('B: Surface Conditions (1000 hPa)', fontsize=14, fontweight='bold')
ax2.set_xlabel('Surface Pressure (Pa)', fontsize=12)
ax2.set_ylabel('24hr Precipitation (mm)', fontsize=12)
ax2.legend(title='Wind Speed (m/s)', loc='upper left', bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.savefig('era5_ff_vnm_analysis.png')
plt.show()

In [ ]:
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import pandas as pd

fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.Robinson()) 

ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.BORDERS, linestyle=':')

plt.scatter(df['longitude'], df['latitude'], c=df['temp_c'], 
            transform=ccrs.PlateCarree(), cmap='RdYlBu_r')

plt.title("Spatial Distribution of Flood Event FF-2004-000060-VNM")
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

file_path = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth/FF_2012_000081_ETH/center_ERA5_truth.csv"
df = pd.read_csv(file_path)

df_surface = df[df['level'] == 1000].copy()

fig, ax = plt.subplots(1, 1, figsize=(10, 7), dpi=300, 
                       subplot_kw={'projection': ccrs.PlateCarree()})

ax.add_feature(cfeature.COASTLINE, linewidth=1)
ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.7)
ax.add_feature(cfeature.RIVERS, linewidth=0.5, edgecolor='blue')
ax.add_feature(cfeature.LAND, facecolor='#f9f9f9')

precip_plot = ax.tricontourf(
    df_surface['longitude'], df_surface['latitude'], 
    df_surface['precipitation_mm_24h'],
    transform=ccrs.PlateCarree(),
    cmap='Blues', 
    levels=15, 
    alpha=0.8
)

plt.colorbar(precip_plot, ax=ax, label='24h Precipitation (mm)', fraction=0.03, pad=0.04)

ax.quiver(
    df_surface['longitude'].values, df_surface['latitude'].values,
    df_surface['u_component_of_wind'].values, df_surface['v_component_of_wind'].values,
    transform=ccrs.PlateCarree(),
    color='black', alpha=0.5, scale=40, width=0.004
)

# Vietnam event center for FF-2004-000060
ax.plot(108.25, 15.25, 'ro', markersize=10, 
        transform=ccrs.PlateCarree(), label='Flood Center', markeredgecolor='white')

ax.set_extent([106.5, 110.0, 13.5, 16.5], crs=ccrs.PlateCarree())

plt.title("ERA5 Ground Truth: FF-2004-000060-VNM\nSurface Precipitation & Wind Vectors", fontsize=14)
plt.legend(loc='upper right')

plt.savefig('vietnam_flood_map.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.ticker as mticker
import numpy as np

file_path = "/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth/FF_2004_000060_VNM/center_ERA5_truth.csv"
df = pd.read_csv(file_path)
df_surface = df[df['level'] == 1000].copy()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 9), dpi=300, 
                               subplot_kw={'projection': ccrs.PlateCarree()})

extent = [107.4, 109.3, 13.7, 15.3]

def format_map(ax, title):
    ax.set_extent(extent, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.COASTLINE, linewidth=1.5, zorder=3)
    ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.7, zorder=3)
    ax.set_aspect('auto')
    
    gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.4, zorder=4)
    gl.top_labels = False
    gl.right_labels = False
    gl.xlabel_style = {'size': 10}
    gl.ylabel_style = {'size': 10}
    ax.set_title(title, fontsize=14, fontweight='bold', pad=15)

format_map(ax1, 'A: Precipitation Distribution')
precip_levels = [0, 2, 5, 10, 20, 30, 40, 50, 55]
cf1 = ax1.tricontourf(df_surface['longitude'], df_surface['latitude'], 
                      df_surface['precipitation_mm_24h'],
                      levels=precip_levels, cmap='YlGnBu', extend='max', zorder=1)

plt.colorbar(cf1, ax=ax1, orientation='horizontal', pad=0.1, fraction=0.05, aspect=30,
             label='24h Accumulated Precipitation (mm)')

format_map(ax2, 'B: Surface Wind Field')
wind_speed = np.sqrt(df_surface['u_component_of_wind']**2 + df_surface['v_component_of_wind']**2)

cf2 = ax2.tricontourf(df_surface['longitude'], df_surface['latitude'], wind_speed,
                      cmap='magma', levels=10, zorder=1)

ax2.quiver(df_surface['longitude'].values, df_surface['latitude'].values,
           df_surface['u_component_of_wind'].values, df_surface['v_component_of_wind'].values,
           color='white', scale=35, width=0.005, zorder=2)

plt.colorbar(cf2, ax=ax2, orientation='horizontal', pad=0.1, fraction=0.05, aspect=30,
             label='10m Wind Speed (m/s)')

plt.suptitle(f"Atmospheric Analysis: FF-2004-000060-VNM", fontsize=18, fontweight='bold', y=1)
plt.tight_layout()
plt.savefig('fixed_dimensions_vnm.png', bbox_inches='tight')
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
import matplotlib.ticker as mticker

def visualize_era5_event(file_path):
    df = pd.read_csv(file_path)
    df_surface = df[df['level'] == 1000].copy()
    
    lon_min, lon_max = df_surface['longitude'].min() - 0.2, df_surface['longitude'].max() + 0.2
    lat_min, lat_max = df_surface['latitude'].min() - 0.2, df_surface['latitude'].max() + 0.2
    auto_extent = [lon_min, lon_max, lat_min, lat_max]

    max_precip = df_surface['precipitation_mm_24h'].max()
    precip_levels = np.linspace(0, max_precip, 11)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8), dpi=300, 
                                   subplot_kw={'projection': ccrs.PlateCarree()})

    def format_map(ax, title):
        ax.set_extent(auto_extent, crs=ccrs.PlateCarree())
        ax.add_feature(cfeature.COASTLINE, linewidth=1.5, zorder=3)
        ax.add_feature(cfeature.BORDERS, linestyle=':', alpha=0.7, zorder=3)
        ax.add_feature(cfeature.LAND, facecolor='#fcfcfc')
        ax.set_aspect('auto') 
        
        gl = ax.gridlines(draw_labels=True, linestyle='--', alpha=0.4, zorder=4)
        gl.top_labels = False
        gl.right_labels = False
        ax.set_title(title, fontsize=14, fontweight='bold', pad=20)

    format_map(ax1, f"A: Precipitation Pattern\nMax: {max_precip:.1f} mm")
    cf1 = ax1.tricontourf(df_surface['longitude'], df_surface['latitude'], 
                          df_surface['precipitation_mm_24h'],
                          levels=precip_levels, cmap='YlGnBu', extend='max')

    plt.colorbar(cf1, ax=ax1, orientation='horizontal', pad=0.12, fraction=0.046, aspect=30,
                 label='24h Accumulated Precipitation (mm)')

    format_map(ax2, "B: Surface Wind Dynamics")
    wind_speed = np.sqrt(df_surface['u_component_of_wind']**2 + df_surface['v_component_of_wind']**2)
    
    cf2 = ax2.tricontourf(df_surface['longitude'], df_surface['latitude'], wind_speed,
                          cmap='magma', levels=10)

    ax2.quiver(df_surface['longitude'].values, df_surface['latitude'].values,
               df_surface['u_component_of_wind'].values, df_surface['v_component_of_wind'].values,
               color='white', scale=None, width=0.005)

    plt.colorbar(cf2, ax=ax2, orientation='horizontal', pad=0.12, fraction=0.046, aspect=30,
                 label='10m Wind Speed (m/s)')

    event_id = df['event_id'].iloc[0]
    region = df['region_label'].iloc[0]
    plt.suptitle(f"Event: {event_id} | Region: {region}", fontsize=18, fontweight='bold', y=0.98)
    
    plt.tight_layout(w_pad=4)
    
    save_name = f"analysis_{event_id}_{region}.png".replace("-", "_")
    plt.savefig(save_name, bbox_inches='tight')
    print(f"Successfully saved: {save_name}")
    plt.show()

visualize_era5_event("/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth/FF_2013_000088_AFG/center_ERA5_truth.csv")
visualize_era5_event("/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth/FF_2021_000098_MNG/Arkhangai_ERA5_truth.csv")
visualize_era5_event("/home/teoaivalis/floods_kg/reanalysis_data/era5_ground_truth/FF_2021_000090_SDN/center_ERA5_truth.csv")

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt

nc_path = '/home/teoaivalis/floods_kg/reanalysis_data/new_pangu_predictions/FL_2022_000378_BIH/center_Pangu_data.nc'
ds = xr.open_dataset(nc_path)

print(ds)

print(f"Levels: {len(ds.level)} (Expected 13)")
print(f"Grid Size: {ds.latitude.size}x{ds.longitude.size} (Expected ~8x8)")

for var in ds.data_vars:
    val_min = ds[var].min().values
    val_max = ds[var].max().values
    nan_count = ds[var].isnull().sum().values
    print(f"{var:25} | Min: {val_min:10.2f} | Max: {val_max:10.2f} | NaNs: {nan_count}")

plt.figure(figsize=(6, 5))
ds['temperature'].sel(level=850).plot()
plt.title(f"850hPa Temperature - {ds.attrs.get('region_label', 'Event Area')}")
plt.show()

In [ ]:
print("Variables in Zarr:", list(ds.data_vars))
print("Max of specific_humidity:", ds.specific_humidity.max().values)

In [ ]:
import numpy as np
import os
from sklearn.metrics.pairwise import cosine_similarity

sig_dir = '/home/teoaivalis/floods_kg/reanalysis_data/event_embeddings'
files = [f for f in os.listdir(sig_dir) if f.endswith('.npy')]

print(f"--- ANALYZING {len(files)} SIGNATURES ---")

all_sigs = []
for f in files[:10]:
    sig = np.load(os.path.join(sig_dir, f))
    all_sigs.append(sig)
    
    has_nan = np.isnan(sig).any()
    v_min, v_max = sig.min(), sig.max()
    v_mean, v_std = sig.mean(), sig.std()
    
    print(f"File: {f[:30]}...")
    print(f"  NaNs: {has_nan} | Range: [{v_min:.3f}, {v_max:.3f}] | Mean: {v_mean:.3f} | Std: {v_std:.3f}")

if len(all_sigs) >= 2:
    print("\n--- SIMILARITY TEST ---")

    sim = cosine_similarity(all_sigs[0].reshape(1, -1), all_sigs[1].reshape(1, -1))[0][0]
    
    print(f"Comparing {files[0]} vs {files[1]}")
    print(f"  Cosine Similarity: {sim:.4f}")

In [ ]:
!pip install microsoft-aurora
!pip install timm einops

In [ ]:
!pip install --upgrade timm

In [ ]:
import timm
print(timm.__version__)

In [ ]:
from datetime import datetime

import torch

from aurora import AuroraSmallPretrained, Batch, Metadata

model = AuroraSmallPretrained()
model.load_checkpoint()

batch = Batch(
    surf_vars={k: torch.randn(1, 2, 17, 32) for k in ("2t", "10u", "10v", "msl")},
    static_vars={k: torch.randn(17, 32) for k in ("lsm", "z", "slt")},
    atmos_vars={k: torch.randn(1, 2, 4, 17, 32) for k in ("z", "u", "v", "t", "q")},
    metadata=Metadata(
        lat=torch.linspace(90, -90, 17),
        lon=torch.linspace(0, 360, 32 + 1)[:-1],
        time=(datetime(2020, 6, 1, 12, 0),),
        atmos_levels=(100, 250, 500, 850),
    ),
)

prediction = model.forward(batch)

print(prediction.surf_vars["2t"])

In [ ]:
from datetime import datetime

import torch

from aurora import AuroraSmallPretrained, Batch, Metadata

model = AuroraSmallPretrained()
model.load_checkpoint()

with torch.no_grad():
    normalized_batch = batch.normalise(model.surf_stats)
    
    device = next(model.parameters()).device
    lead_time = torch.tensor([0.0], dtype=torch.float32).to(device)

    latent_tokens = model.model.encoder(normalized_batch, lead_time)

    knowledge_state = model.model.backbone(latent_tokens)

signature = torch.mean(knowledge_state, dim=1).squeeze().cpu().numpy()

print("--- FOUNDATION KNOWLEDGE SIGNATURE ---")
print(f"Vector Length: {len(signature)}")
print(f"First 5 values: {signature[:5]}")
print(f"Vector Variance: {signature.var():.4f}")